In [ ]:
import anndata
import numpy as np
import scvelo as scv
import scanpy as sc
import torch
import velovae as vv
import pickle as pickle
import matplotlib.pyplot as plt
import time
import pandas as pd
import unitvelo as utv

adata = sc.read_h5ad('../Data/simulated_T_cell.h5ad')
adata.obsm['X_umap'] = pickle.load(open(f'../Data/UMAP.pkl','rb'))
scv.pp.moments(adata, n_neighbors=30)

vae = vv.VAE(adata, tmax=20, dim_z=5, device='cuda', full_vb=True)
vae.train(adata, plot=False, figure_path='../Results/VeloVAE', embed='umap')

vae.save_anndata(adata, 'vae', '../Results/VeloVAE/', file_name="VeloVAE.h5ad")
adata1 = sc.read_h5ad('../Results/VeloVAE//VeloVAE.h5ad')

scv.tl.velocity_graph(adata1, vkey='vae_velocity', n_jobs=-1)
scv.tl.velocity_embedding(adata1, basis='umap', vkey='vae_velocity')
fix, ax = plt.subplots(1, 1, figsize = (8, 6))
scv.pl.velocity_embedding_stream(adata1, basis='umap', save = False, vkey='vae_velocity', color='time', title='VeloVAE', fontsize=20, show = False, ax = ax, legend_fontsize = 15)
plt.savefig('../Results/VeloVAE_UMAP_stream.png', dpi=80)